In [ ]:
# ============ 用户可调参数（运行前先确认） ============
# 项目目录必须包含 03_results/run_manifest.json；Pipeline 目录必须包含
# downstream/downstream_qc.py 和 core/run_pipeline.sh。
# Pipeline 和项目通常不在同一目录，请直接填写绝对路径。
DNA_PROJECT_ROOT = None    # 例如 Path("/absolute/path/to/Patient001")
DNA_PIPELINE_ROOT = None   # 例如 Path("/absolute/path/to/Alopex")

# 默认按项目 species 读取 Pipeline 内 TSS。只有需要覆盖时填写绝对路径；
# 没有标准 TSS 资源的非模式物种可设置 SKIP_TSS=True。
TSS_BED = None             # 例如 Path("/absolute/path/to/genome_TSS_2000_2000_20.bed")
SKIP_TSS = False

import json
import os
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_pipeline_root() -> Path:
    # downstream_qc.py 位于 Pipeline 树内，定位它的引导链必须留在 Notebook；
    # 完整契约（显式不回退、schema/status/source_root 校验）以
    # downstream_qc.resolve_notebook_paths 为唯一实现。
    option = str(DNA_PIPELINE_ROOT or os.environ.get("DNA_PIPELINE_ROOT") or "").strip()
    if option:
        return Path(option).expanduser().resolve()
    project_option = str(DNA_PROJECT_ROOT or os.environ.get("DNA_PROJECT_ROOT") or "").strip()
    bases = [Path(project_option).expanduser().resolve()] if project_option else []
    start = Path.cwd().resolve()
    bases.extend([start, *start.parents])
    for base in bases:
        manifest = base / "03_results" / "run_manifest.json"
        if manifest.is_file():
            payload = json.loads(manifest.read_text(encoding="utf-8"))
            source_root = (
                payload.get("pipeline", {}).get("source_root")
                if isinstance(payload, dict)
                else None
            )
            if not isinstance(source_root, str) or not source_root.strip():
                raise FileNotFoundError(
                    f"run_manifest.pipeline.source_root is missing: {manifest}"
                )
            return Path(source_root).expanduser().resolve()
    for base in [start, *start.parents]:
        if (base / "downstream" / "downstream_qc.py").is_file() and (
            base / "core" / "run_pipeline.sh"
        ).is_file():
            return base
    raise FileNotFoundError("Cannot locate the Alopex source tree; set DNA_PIPELINE_ROOT.")


_BOOTSTRAP_ROOT = _bootstrap_pipeline_root()
if not (_BOOTSTRAP_ROOT / "downstream" / "downstream_qc.py").is_file() or not (
    _BOOTSTRAP_ROOT / "core" / "run_pipeline.sh"
).is_file():
    raise FileNotFoundError(f"Not a usable Alopex: {_BOOTSTRAP_ROOT}")
_BOOTSTRAP_DOWNSTREAM = str(_BOOTSTRAP_ROOT / "downstream")
if _BOOTSTRAP_DOWNSTREAM not in sys.path:
    sys.path.insert(0, _BOOTSTRAP_DOWNSTREAM)

from downstream_qc import (
    load_cpg_density_frame,
    load_notebook_qc_frame,
    make_clone_selection_suffix,
    plot_cpg_density,
    plot_cpg_signal_composition,
    plot_final_qc_dashboard,
    plot_final_qc_dashboard_batch,
    plot_pca_batch_effect,
    plot_plate_metric_map,
    plot_tss_profile,
    plot_wgbs_qc_panel,
    prepare_single_cpg_adata,
    resolve_notebook_paths,
    run_qc_processor,
)

PATHS = resolve_notebook_paths(DNA_PROJECT_ROOT, DNA_PIPELINE_ROOT)
PROJECT_DIR = PATHS.project_dir
PIPELINE_PATH = PATHS.pipeline_root

print(f"Pipeline: {PIPELINE_PATH}")
print(f"Project: {PROJECT_DIR}")
print(f"QC results: {PATHS.qc_results_dir}")


In [ ]:
# 强制重算 / 重建开关；默认 False 时按内容与缓存身份自动判定复用
RECOMPUTE_RAW_ADATA = False
RECOMPUTE_METHYL_STATS = False
RECOMPUTE_COMPOSITION = False
RECOMPUTE_GINI = False
RECOMPUTE_TSS = False

run_qc_processor(
    PATHS,
    tss_bed=TSS_BED,
    skip_tss=SKIP_TSS,
    recompute_raw_adata=RECOMPUTE_RAW_ADATA,
    recompute_methyl_stats=RECOMPUTE_METHYL_STATS,
    recompute_composition=RECOMPUTE_COMPOSITION,
    recompute_gini=RECOMPUTE_GINI,
    recompute_tss=RECOMPUTE_TSS,
)


In [ ]:
df_qc, clone_color_map = load_notebook_qc_frame(PATHS)

PLOTS_DIR = PATHS.plots_dir
QC_INFO_PATH = PATHS.qc_info
TSS_INFO_PATH = PATHS.tss_info
COMPOSITION_CACHE_PATH = PATHS.composition_cache


In [ ]:
df_qc

In [ ]:
df_qc.groupby("CloneID").median(numeric_only=True)

In [ ]:
df_qc.groupby("CloneID").mean(numeric_only=True)

In [ ]:
MAPPING_THRESHOLD = None
mapping_hq = True if MAPPING_THRESHOLD is None else (df_qc['Native_Mapping%'] > MAPPING_THRESHOLD)
df_qc['HQ'] = (
    mapping_hq &
    (df_qc['Lambda_Meth_CpG_Rate%'] < 10) &
    (df_qc['pUC19_Meth_CpG_Rate%'] > 90) &
    (df_qc['Sample_Unique_CpG_Sites'] > 500000) &
    (df_qc['Gini_Index'] < 0.5)
)

df_qc.groupby('CloneID')['HQ'].mean()*100

In [ ]:
if not df_qc.empty:
    plot_wgbs_qc_panel(df_qc, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'WGBS_QC_Panel.pdf')


## 最终 QC 指标说明

- **Native_Mapping%**：backend 原生 mapping 率。BISCUIT 为 primary `MAPQ>=40` 的 individual reads，Bismark 为 unique concordant read pairs；两种口径单位不同，不能直接跨 backend 比较。
- **Duplicate_Pair_Rate%**：duplicate pairs / backend accepted pairs。
- **Final_Pair_Yield%**：最终保留的 read pairs / Cutadapt input pairs。

两个 backend 的 mapping policy 不同，默认不用统一的 mapping 阈值参与 HQ 判定；如项目已有经过验证的阈值，可显式设置 `MAPPING_THRESHOLD`。

In [ ]:
FINAL_QC_MODE = 'per_clone_samples'

FINAL_QC_CLONE_ID = None # 可以选择样品范围

FINAL_QC_METRIC = [
    'Sample_Unique_CpG_Sites',
    'Native_Mapping%',
    'Final_Pair_Yield%',
    'Duplicate_Pair_Rate%',
    'Non_CpG_Methylation%',
]

FINAL_QC_SIMPLIFY_SAMPLE_ID = True

if not df_qc.empty:
    if FINAL_QC_MODE == 'per_clone_samples' and isinstance(FINAL_QC_METRIC, (list, tuple)):
        plot_final_qc_dashboard_batch(
            df_qc,
            metrics=FINAL_QC_METRIC,
            clone_id=FINAL_QC_CLONE_ID,
            clone_color_map=clone_color_map,
            out_dir=PLOTS_DIR,
            filename_prefix='Final_QC_Dashboard',
            mode=FINAL_QC_MODE,
            simplify_sample_ids=FINAL_QC_SIMPLIFY_SAMPLE_ID,
        )
    else:
        clone_suffix = make_clone_selection_suffix(FINAL_QC_CLONE_ID)
        metric_suffix = FINAL_QC_METRIC if FINAL_QC_MODE == 'per_clone_samples' else 'all_metrics'
        plot_final_qc_dashboard(
            df_qc,
            clone_id=FINAL_QC_CLONE_ID,
            mode=FINAL_QC_MODE,
            metric=FINAL_QC_METRIC,
            clone_color_map=clone_color_map,
            out_path=PLOTS_DIR / f'Final_QC_Dashboard_{FINAL_QC_MODE}_{metric_suffix}_{clone_suffix}.pdf',
            simplify_sample_ids=FINAL_QC_SIMPLIFY_SAMPLE_ID,
        )


In [ ]:
PLATE_VIEW_METRIC = 'Native_Mapping%' #Sample_Unique_CpG_Sites

if not df_qc.empty:
    plot_plate_metric_map(
        df_qc,
        metric=PLATE_VIEW_METRIC,
        clone_color_map=clone_color_map,
        out_path=PLOTS_DIR / f'Plate_View_{PLATE_VIEW_METRIC}.pdf',
    )

In [ ]:
SIGNAL_COMPOSITION_SIMPLIFY_SAMPLE_ID = True

if not df_qc.empty:
    plot_cpg_signal_composition(
        df_qc,
        out_path=PLOTS_DIR / 'CpG_Signal_Composition.pdf',
        simplify_sample_ids=SIGNAL_COMPOSITION_SIMPLIFY_SAMPLE_ID,
    )


In [ ]:
if not SKIP_TSS and TSS_INFO_PATH.exists():
    plot_tss_profile(TSS_INFO_PATH, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'TSS_Profile.pdf')
elif SKIP_TSS:
    print('已按 SKIP_TSS 跳过 TSS；该交付的旧 TSS CSV/PDF 已清理。')
else:
    print('⚠️ 未生成 TSS Profile 数据；请检查当前 species 的 Pipeline TSS 资源，或显式设置 TSS_BED。')


In [ ]:
if not df_qc.empty:
    df_density = load_cpg_density_frame(
        PROJECT_DIR,
        QC_INFO_PATH,
        COMPOSITION_CACHE_PATH,
    )
    plot_cpg_density(df_density, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'CpG_Density.pdf')


In [ ]:
if not df_qc.empty:
    plot_pca_batch_effect(df_qc, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'Batch_Effect_PCA.pdf')


## 生成并保存 single-CpG 矩阵

运行最后一格，从本次交付的 RawAdata 生成 `SingleCpG_Adata.h5ad`。使用当前物种的 single-CpG BED，参数为 `fraction`、`mean`、`chunk_size=500`。保留全部细胞，按 `Sample_ID` 对齐并保存当前 `df_qc` 的 QC、`HQ` 及额外注释列；此处不按 HQ 筛选。

首次运行需要完成矩阵计算；重复运行复用已验证的矩阵，仅注释变化时更新 `obs`。来源改变或现有文件未经验证时会提示，确认需要重建后将 `RECOMPUTE_SINGLECPG` 改为 `True`。完成后输入和输出文件均关闭，下一次运行请恢复 `False`。

下游直接使用打印出的 `SINGLECPG_ADATA_PATH`。`snap.read(SINGLECPG_ADATA_PATH, backed="r")` 适用于读取；该对象不能直接回填标签。需要修改时使用独立的可写副本或能容纳的内存子集。这里只提前生成矩阵，MethylTree 分箱后仍须按细胞 ID 保留或恢复 obs。

In [ ]:
RECOMPUTE_SINGLECPG = False

SINGLECPG_ADATA_PATH = prepare_single_cpg_adata(
    PATHS, df_qc, recompute=RECOMPUTE_SINGLECPG,
)
print(f"SINGLECPG_ADATA_PATH = {str(SINGLECPG_ADATA_PATH)!r}")